In [0]:
%pip install sqlglot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.0/719.0 kB 23.9 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import sqlglot
from sqlglot import parse_one
from sqlglot import expressions as exp

from pathlib import Path
from datetime import datetime

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS mdo_dev_dbx.metadata;

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/mdo_dev_dbx/metadata/sql_versions/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/mdo_dev_dbx/metadata/sql_versions/customer_revenue_v1.sql,customer_revenue_v1.sql,1105,1785309373000
dbfs:/Volumes/mdo_dev_dbx/metadata/sql_versions/customer_revenue_v2.sql,customer_revenue_v2.sql,1161,1785309373000


In [0]:
%sql
SHOW VOLUMES IN mdo_dev_dbx.metadata;

database,volume_name
metadata,sql_versions


In [0]:
sql_v1 = dbutils.fs.head(
    "/Volumes/mdo_dev_dbx/metadata/sql_versions/customer_revenue_v1.sql"
)

sql_v2 = dbutils.fs.head(
    "/Volumes/mdo_dev_dbx/metadata/sql_versions/customer_revenue_v2.sql"
)

Parse SQL

In [0]:
ast_v1 = parse_one(
    sql_v1,
    read="spark"
)


ast_v2 = parse_one(
    sql_v2,
    read="spark"
)
print(ast_v1)

WITH customer_sales AS (SELECT ct.customer_id, ct.product_id, p.product_category, c.region, c.customer_segment, ct.transaction_date, ct.quantity, ct.total_sales, ct.discount, ct.profit FROM customer_transactions AS ct INNER JOIN products AS p ON ct.product_id = p.product_id INNER JOIN customers AS c ON ct.customer_id = c.customer_id WHERE ct.transaction_status = 'Completed' AND ct.transaction_date >= CAST('2024-01-01' AS DATE)), regional_sales AS (SELECT product_category, region, customer_segment, SUM(total_sales) AS revenue, SUM(profit) AS total_profit, SUM(quantity) AS units_sold, COUNT(DISTINCT customer_id) AS active_customers, AVG(total_sales) AS avg_order_value FROM customer_sales GROUP BY product_category, region, customer_segment) SELECT * FROM regional_sales ORDER BY revenue DESC


SQL semantic extractor

In [0]:
def extract_sql_metadata(ast):

    metadata = {}

    # Physical tables only
    cte_names = {
        c.alias
        for c in ast.find_all(exp.CTE)
    }

    metadata["tables"] = sorted(
        {
            table.name
            for table in ast.find_all(exp.Table)
            if table.name not in cte_names
        }
    )


    # Unique columns
    metadata["columns"] = sorted(
        {
            col.sql()
            for col in ast.find_all(exp.Column)
        }
    )


    # Business functions only
    ignore_functions = {
        "AND",
        "OR",
        "CAST"
    }


    metadata["functions"] = sorted(
        {
            func.sql_name().upper()
            for func in ast.find_all(exp.Func)
            if func.sql_name().upper()
            not in ignore_functions
        }
    )


    # Aggregation expressions
    metadata["aggregations"] = sorted(
        {
            agg.sql()
            for agg in ast.find_all(exp.AggFunc)
        }
    )


    # Join types
    joins=[]

    for join in ast.find_all(exp.Join):

        kind=join.args.get("kind")

        if kind is None:
            kind="INNER"

        joins.append(
            str(kind).upper()
        )


    metadata["joins"]=sorted(set(joins))


    # WHERE clauses
    metadata["filters"] = sorted(
        {
            where.this.sql()
            for where in ast.find_all(exp.Where)
        }
    )


    # Group By
    metadata["group_by"] = sorted(
        {
            group.sql()
            for group in ast.find_all(exp.Group)
        }
    )


    return metadata

In [0]:
def read_sql_file(path):

    return dbutils.fs.head(
        path,
        100000
    )

In [0]:
sql_v1 = read_sql_file('/Volumes/mdo_dev_dbx/metadata/sql_versions/customer_revenue_v1.sql')

sql_v2 = read_sql_file('/Volumes/mdo_dev_dbx/metadata/sql_versions/customer_revenue_v2.sql')


print("SQL VERSION 1")
print(sql_v1)


print("----------------")


print("SQL VERSION 2")
print(sql_v2)

SQL VERSION 1
WITH customer_sales AS (

    SELECT

        ct.customer_id,
        ct.product_id,
        p.product_category,
        c.region,
        c.customer_segment,
        ct.transaction_date,

        ct.quantity,
        ct.total_sales,
        ct.discount,
        ct.profit

    FROM customer_transactions ct

    INNER JOIN products p
        ON ct.product_id = p.product_id

    INNER JOIN customers c
        ON ct.customer_id = c.customer_id

    WHERE
        ct.transaction_status = 'Completed'
        AND ct.transaction_date >= DATE('2024-01-01')
),

regional_sales AS (

    SELECT

        product_category,

        region,

        customer_segment,

        SUM(total_sales) AS revenue,

        SUM(profit) AS total_profit,

        SUM(quantity) AS units_sold,

        COUNT(DISTINCT customer_id) AS active_customers,

        AVG(total_sales) AS avg_order_value

    FROM customer_sales

    GROUP BY

        product_category,
        region,
        customer_segment



In [0]:
metadata_v1 = extract_sql_metadata(ast_v1)

metadata_v2 = extract_sql_metadata(ast_v2)
for key,value in metadata_v1.items():

    print("\n======",key,"======")

    print("V1:")
    print(value)

    print("V2:")
    print(metadata_v2[key])


====== tables ======
V1:
['customer_transactions', 'customers', 'products']
V2:
['customer_transactions', 'customers', 'products']

====== columns ======
V1:
['c.customer_id', 'c.customer_segment', 'c.region', 'ct.customer_id', 'ct.discount', 'ct.product_id', 'ct.profit', 'ct.quantity', 'ct.total_sales', 'ct.transaction_date', 'ct.transaction_status', 'customer_id', 'customer_segment', 'p.product_category', 'p.product_id', 'product_category', 'profit', 'quantity', 'region', 'revenue', 'total_sales']
V2:
['c.customer_id', 'c.customer_segment', 'c.region', 'ct.customer_id', 'ct.discount', 'ct.product_id', 'ct.profit', 'ct.quantity', 'ct.total_sales', 'ct.transaction_date', 'ct.transaction_status', 'customer_id', 'customer_segment', 'net_sales', 'p.product_category', 'p.product_id', 'product_category', 'profit', 'quantity', 'region', 'revenue']

====== functions ======
V1:
['AVG', 'COUNT', 'SUM']
V2:
['AVG', 'COUNT', 'SUM']

====== aggregations ======
V1:
['AVG(total_sales)', 'COUNT(DIST

Compare semantic differences

In [0]:
def compare_component(old,new):

    old=set(old)
    new=set(new)

    return {

        "added": list(new-old),

        "removed": list(old-new),

        "changed": list(old.symmetric_difference(new))
    }

In [0]:
sql_diff={}


for component in metadata_v1:

    sql_diff[component]=compare_component(
        metadata_v1[component],
        metadata_v2[component]
    )


sql_diff

{'tables': {'added': [], 'removed': [], 'changed': []},
 'columns': {'added': ['net_sales'],
  'removed': ['total_sales'],
  'changed': ['total_sales', 'net_sales']},
 'functions': {'added': [], 'removed': [], 'changed': []},
 'aggregations': {'added': ['AVG(net_sales)',
   'SUM(net_sales)',
   'COUNT(customer_id)',
   'AVG(profit)'],
  'removed': ['SUM(profit)',
   'AVG(total_sales)',
   'COUNT(DISTINCT customer_id)',
   'SUM(total_sales)'],
  'changed': ['SUM(net_sales)',
   'AVG(net_sales)',
   'AVG(total_sales)',
   'SUM(profit)',
   'SUM(total_sales)',
   'COUNT(customer_id)',
   'AVG(profit)',
   'COUNT(DISTINCT customer_id)']},
 'joins': {'added': [], 'removed': [], 'changed': []},
 'filters': {'added': ["ct.transaction_status = 'Completed' AND ct.transaction_date >= CAST('2024-06-01' AS DATE) AND c.region <> 'Unknown'",
   'revenue > 1000'],
  'removed': ["ct.transaction_status = 'Completed' AND ct.transaction_date >= CAST('2024-01-01' AS DATE)"],
  'changed': ["ct.transaction_

SQL similarity scoring engine

In [0]:
def compare_component(old,new):

    old=set(old)
    new=set(new)

    return {

        "added": list(new-old),

        "removed": list(old-new),

        "unchanged": list(old & new)

    }
semantic_diff={}

for component in metadata_v1:

    semantic_diff[component]=compare_component(
        metadata_v1[component],
        metadata_v2[component]
    )


semantic_diff

{'tables': {'added': [],
  'removed': [],
  'unchanged': ['products', 'customer_transactions', 'customers']},
 'columns': {'added': ['net_sales'],
  'removed': ['total_sales'],
  'unchanged': ['c.region',
   'ct.customer_id',
   'ct.quantity',
   'c.customer_id',
   'ct.total_sales',
   'customer_segment',
   'revenue',
   'c.customer_segment',
   'region',
   'ct.discount',
   'quantity',
   'customer_id',
   'ct.product_id',
   'ct.transaction_status',
   'p.product_category',
   'ct.transaction_date',
   'product_category',
   'profit',
   'p.product_id',
   'ct.profit']},
 'functions': {'added': [],
  'removed': [],
  'unchanged': ['COUNT', 'AVG', 'SUM']},
 'aggregations': {'added': ['AVG(net_sales)',
   'SUM(net_sales)',
   'COUNT(customer_id)',
   'AVG(profit)'],
  'removed': ['SUM(profit)',
   'AVG(total_sales)',
   'COUNT(DISTINCT customer_id)',
   'SUM(total_sales)'],
  'unchanged': ['SUM(quantity)']},
 'joins': {'added': [], 'removed': [], 'unchanged': ['INNER']},
 'filters':

Similarity Score

In [0]:
def calculate_similarity(metadata1, metadata2):

    scores=[]

    for component in metadata1:

        old=set(metadata1[component])

        new=set(metadata2[component])


        union=len(old.union(new))


        if union==0:
            score=100

        else:

            score=(
                len(old.intersection(new))
                /
                union
            )*100


        scores.append(score)


    return round(
        sum(scores)/len(scores),
        2
    )

In [0]:
similarity_score = calculate_similarity(
    metadata_v1,
    metadata_v2
)


print(
    f"SQL Similarity: {similarity_score}%"
)

SQL Similarity: 71.72%


In [0]:
def generate_findings(diff):

    findings=[]


    if diff["aggregations"]["added"]:

        findings.append(
            "Revenue calculation modified"
        )


    if diff["filters"]["added"]:

        findings.append(
            "WHERE condition changed"
        )


    if diff["joins"]["added"]:

        findings.append(
            "Join logic changed"
        )


    if diff["columns"]["removed"]:

        findings.append(
            "Column removed"
        )


    return findings
findings = generate_findings(
    semantic_diff
)


for f in findings:
    print("-",f)

- Revenue calculation modified
- WHERE condition changed
- Column removed


In [0]:
def calculate_sql_risk(diff):

    findings = []

    risk_score = 0


    # Aggregation changes
    if (
        diff["aggregations"]["added"]
        or diff["aggregations"]["removed"]
    ):

        findings.append({

            "changed_component":
            "Aggregation Logic",

            "severity":
            "HIGH",

            "score":
            40,

            "description":
            "Business metric aggregation changed"

        })

        risk_score += 40



    # Join changes
    if (
        diff["joins"]["added"]
        or diff["joins"]["removed"]
    ):

        findings.append({

            "changed_component":
            "Join Logic",

            "severity":
            "HIGH",

            "score":
            30,

            "description":
            "Join relationship changed"

        })

        risk_score += 30



    # Filter changes
    if diff["filters"]["added"]:

        findings.append({

            "changed_component":
            "Filter Condition",

            "severity":
            "MEDIUM",

            "score":
            20,

            "description":
            "Business filtering criteria changed"

        })

        risk_score += 20



    # Column removal
    if diff["columns"]["removed"]:

        findings.append({

            "changed_component":
            "Column Removal",

            "severity":
            "HIGH",

            "score":
            30,

            "description":
            "Existing column removed"

        })

        risk_score += 30


    return findings, min(risk_score,100)

In [0]:
risk_findings, risk_score = calculate_sql_risk(
    semantic_diff
)

In [0]:
print("Risk Score:", risk_score)


for item in risk_findings:

    print(
        item["severity"],
        "-",
        item["changed_component"],
        "-",
        item["description"]
    )

Risk Score: 90
HIGH - Aggregation Logic - Business metric aggregation changed
MEDIUM - Filter Condition - Business filtering criteria changed
HIGH - Column Removal - Existing column removed


SQL Drift Delta Records

In [0]:
from datetime import datetime


sql_drift_records = []


for item in risk_findings:

    sql_drift_records.append({

        "run_date":
        datetime.now(),

        "sql_file":
        "customer_revenue_v2.sql",

        "change_type":
        "METRIC_LOGIC_CHANGE",

        "changed_component":
        item["changed_component"],

        "similarity_score":
        similarity_score,

        "risk_score":
        risk_score,

        "severity":
        item["severity"],

        "description":
        item["description"]

    })

In [0]:
sql_drift_results_df = spark.createDataFrame(
    sql_drift_records
)


display(sql_drift_results_df)

change_type,changed_component,description,risk_score,run_date,severity,similarity_score,sql_file
METRIC_LOGIC_CHANGE,Aggregation Logic,Business metric aggregation changed,90,2026-07-29T09:06:33.276061Z,HIGH,71.72,customer_revenue_v2.sql
METRIC_LOGIC_CHANGE,Filter Condition,Business filtering criteria changed,90,2026-07-29T09:06:33.276068Z,MEDIUM,71.72,customer_revenue_v2.sql
METRIC_LOGIC_CHANGE,Column Removal,Existing column removed,90,2026-07-29T09:06:33.276072Z,HIGH,71.72,customer_revenue_v2.sql


In [0]:
(
    sql_drift_results_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "mdo_dev_dbx.default.sql_drift_results_delta"
    )
)